# SQLBuddy 04 - Live demo on a Kaggle GPU (fast backup for the presentation)
Runs **the latest SQLBuddy app from GitHub** (the code is downloaded from the `main` branch at the
start of every run, so every merged design change shows up here automatically; the fine-tuned adapter
comes from the public Hugging Face model repo) on a T4 GPU: answers take ~1-4 s instead of 3-15 s on a laptop CPU.

**Use:** open this notebook in the Kaggle editor -> right panel: *Accelerator = GPU T4*, *Internet = on*
-> **Run all** -> after ~3-5 min the last cell prints `PUBLIC LINK: https://....gradio.live`.
The link works for anyone for `DEMO_MINUTES` (90). Press *Stop* when you are done to save GPU quota.

In [ ]:
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 huggingface_hub==2.4.0 nltk==3.9.2 gradio==6.31.0

In [ ]:
# Download the latest app from GitHub (branch main): the single source of truth for the demo code.
import io, json, os, shutil, sys, urllib.request, zipfile
REPO, BRANCH = "MIbrahimSalman/sqlbuddy", "main"
sha = json.load(urllib.request.urlopen(f"https://api.github.com/repos/{REPO}/commits/{BRANCH}"))["sha"]
data = urllib.request.urlopen(f"https://github.com/{REPO}/archive/{sha}.zip").read()
shutil.rmtree("/kaggle/working/sqlbuddy", ignore_errors=True)
zf = zipfile.ZipFile(io.BytesIO(data))
zf.extractall("/kaggle/working")
work = os.path.join("/kaggle/working", zf.namelist()[0].rstrip("/"))   # sqlbuddy-<sha>/
os.rename(work, "/kaggle/working/sqlbuddy"); work = "/kaggle/working/sqlbuddy"
print("app version: github.com/%s commit %s" % (REPO, sha[:7]))
sys.path.insert(0, os.path.join(work, "app"))

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (enable GPU T4!)")
import app as sqlbuddy_app
sqlbuddy_app.MODELS.load()                 # base model + public LoRA adapter, on the GPU
sqlbuddy_app.MODELS.warm_up(sqlbuddy_app.schema_of(sqlbuddy_app.SAMPLE_DB))   # cache the sample schema prompt
print("device:", sqlbuddy_app.MODELS.describe(), "| adapter loaded:", sqlbuddy_app.MODELS.has_adapter,
      sqlbuddy_app.MODELS.adapter_error or "")

In [ ]:
# Self-test: one example question through both models (same code path as the UI button)
q = "What is the average score in each course? Show the course title."
db = str(sqlbuddy_app.SAMPLE_DB)
for ft in (False, True):
    sql, df, status = sqlbuddy_app.answer_one(db, q, ft)
    print(status)
    print("  ", sql)
    print(df.head(3))
    print()

In [ ]:
import time
# Started from the editor ("Interactive") -> 90 min demo; pushed from the CLI ("Batch") -> quick check.
RUN_TYPE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive")
DEMO_MINUTES = 1 if RUN_TYPE == "Batch" else 90
print("run type:", RUN_TYPE, "| demo minutes:", DEMO_MINUTES)
demo = sqlbuddy_app.build_ui()
demo.queue(default_concurrency_limit=8).launch(share=True, prevent_thread_lock=True, **sqlbuddy_app.LAUNCH_KW)
print("PUBLIC LINK:", demo.share_url)      # open this *.gradio.live URL in any browser
time.sleep(DEMO_MINUTES * 60)
demo.close()